<a href="https://colab.research.google.com/github/edilsonuamusse/kiva_agricultural_loans/blob/main/kiva_agricultural_loans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
%pip -q install kagglehub

import kagglehub
import pandas as pd

# Download the CSV from Kaggle
path = kagglehub.dataset_download(
    "kiva/data-science-for-good-kiva-crowdfunding",
    path="kiva_loans.csv"
)

# Read the CSV and select agricultural loans
loans = pd.read_csv(path)
agricultural_loans = loans[loans["sector"] == "Agriculture"]

print("Total loans:", len(loans))
print("Agricultural loans:", len(agricultural_loans))

Using Colab cache for faster access to the 'data-science-for-good-kiva-crowdfunding' dataset.
Total loans: 671205
Agricultural loans: 180302


# In which countries was at least one agricultural loan fully funded?

In [19]:
fully_funded = agricultural_loans["funded_amount"] == agricultural_loans["loan_amount"]
funded_loans = agricultural_loans[fully_funded]
countries = funded_loans.groupby("country").size().reset_index(name="count")

print("Number of countries:", len(countries))
for country in countries["country"]:
    print(country)

Number of countries: 77
Albania
Armenia
Azerbaijan
Belize
Benin
Bolivia
Brazil
Burkina Faso
Burundi
Cambodia
Cameroon
China
Colombia
Congo
Costa Rica
Cote D'Ivoire
Dominican Republic
Ecuador
Egypt
El Salvador
Georgia
Ghana
Guatemala
Haiti
Honduras
India
Indonesia
Iraq
Israel
Jordan
Kenya
Kosovo
Kyrgyzstan
Lebanon
Liberia
Madagascar
Malawi
Mali
Mexico
Moldova
Mongolia
Mozambique
Myanmar (Burma)
Nepal
Nicaragua
Nigeria
Pakistan
Palestine
Panama
Paraguay
Peru
Philippines
Puerto Rico
Rwanda
Saint Vincent and the Grenadines
Samoa
Senegal
Sierra Leone
Solomon Islands
Somalia
South Africa
South Sudan
Suriname
Tajikistan
Tanzania
Thailand
The Democratic Republic of the Congo
Timor-Leste
Togo
Turkey
Uganda
Ukraine
United States
Vietnam
Yemen
Zambia
Zimbabwe


In 77 countries in the Kiva dataset, at least one agricultural loan recieved the full amount requested.

# What was the median loan amount requested for each agricultural activity, and which activity had the highest median?

In [20]:
median_by_activity = agricultural_loans.groupby("activity")["loan_amount"].median()
median_by_activity.sort_values(ascending=False)

,loan_amount
activity,
Beekeeping,1500.0
Cattle,825.0
Livestock,825.0
Land Rental,675.0
Agriculture,650.0
Animal Sales,625.0
Farm Supplies,600.0
Veterinary Sales,550.0
Aquaculture,512.5


Beekeeping had the highest median loan amount requested, at 1500. Cattle and Livestock were tied for seecond, with a median of $825 each

# Within each requested loan amount range, what percentage of agricultural loans were fully funded?

In [21]:
loans["amount_range"] = loans["loan_amount"].apply(
    lambda amount: "1. Up to $500" if amount <= 500
    else "2. Over $500 to $1,000" if amount <= 1000
    else "3. Over $1,000"
)

loans["fully_funded"] = loans["funded_amount"] == loans["loan_amount"]

agricultural_loans = loans[loans["sector"] == "Agriculture"]
funded_percentage = agricultural_loans.groupby("amount_range")["fully_funded"].mean() * 100
funded_percentage

,fully_funded
amount_range,
1. Up to $500,97.391408
"2. Over $500 to $1,000",89.964272
"3. Over $1,000",83.681804


In the Kiva dataset, 97.4% of agricultural loans of $500 or less were fully funded. This fell to 90% for requests over 500 and up to 1,000 and 83.7% for requests over 1000

#

# Across all years in the dataset. which month or months had the most agricultural loan requests in Mozambique?

In [22]:
loans["month"] = loans["posted_time"].apply(lambda date: date[5:7])

agricultural_loans = loans[loans["sector"] == "Agriculture"]
mozambique_loans = agricultural_loans[agricultural_loans["country"] == "Mozambique"]

requests_by_month = mozambique_loans.groupby("month").size()
requests_by_month.sort_values(ascending=False)

,0
month,
06,50
12,45
07,39
03,37
08,35
05,28
04,27
09,23
01,22


Across all years in the dataset, June had the most agricultural loan requests in Mozambique, with 50 out of 346 requests (14.5%). December ranked second with 45 requests, followed by July with 39.

# How did the median amount requested for agricultural loans in Mozambique change from year to year?

In [16]:
loans["year"] = loans["posted_time"].apply(lambda date: date[:4])

agricultural_loans = loans[loans["sector"] == "Agriculture"]
mozambique_loans = agricultural_loans[agricultural_loans["country"] == "Mozambique"]

median_by_year = mozambique_loans.groupby("year")["loan_amount"].median()
median_by_year

,loan_amount
year,
2014,812.5
2015,750.0
2016,350.0
2017,512.5


In Mozambique, the median amount requested for agricultural loans fell from 812.50 in 2014 to 750 in 2015 and 350 in 2016. It rose to $512.50 in 2017, although the last request recorded for that year was on 5 July, so the 2017 figure covers only part of the year.

# In Mozambique, how did the median number of lenders compare between agricultural loans that mentioned buying seeds and those that did not?

In [17]:
mozambique_loans = agricultural_loans[agricultural_loans["country"] == "Mozambique"]
descriptions = mozambique_loans["use"]

mentions_seeds = descriptions.str.contains(r"\bseeds?\b", case=False)
mentions_purchase = descriptions.str.contains("buy|purchase|acquire|pay for", case=False)
seed_purchase = mentions_seeds & mentions_purchase

print("Mentions buying seeds:", mozambique_loans[seed_purchase == True]["lender_count"].median())
print("Does not mention buying seeds:", mozambique_loans[seed_purchase == False]["lender_count"].median())

Mentions buying seeds: 23.0
Does not mention buying seeds: 16.5


In Mozambique, the 54 agricultural loans whose descriptions mentioned buying seeds had a median of 23 lenders, compared with 16 for the 283 loans whose descriptions did not mention buying seeds. The median was therefore seven lenders higher for loans that mentioned buying seeds.